##Read Bronze Data

In [0]:
#make changes

In [0]:
df = (
    spark.read
    .option("multiline", "true")
    .json(
        "s3://weather-data-pipeline-235/bronze/weather/"
    )
)


## Import Required Functions

In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    to_date,
    when
)

##Standardize Data Types

In [0]:
df = (
    df
    .withColumn(
        "humidity",
        col("humidity").cast("integer")
    )
    .withColumn(
        "pressure",
        col("pressure").cast("integer")
    )
    .withColumn(
        "temperature",
        col("temperature").cast("double")
    )
    .withColumn(
        "wind_speed",
        col("wind_speed").cast("double")
    )
)


##Create Timestamp Column

In [0]:
df = (
    df
    .withColumn(
        "ingestion_ts",
        to_timestamp(
            "ingestion_timestamp"
        )
    )
)


##create date column

In [0]:
df = (
    df
    .withColumn(
        "ingestion_date",
        to_date(
            "ingestion_ts"
        )
    )
)


## Create temperature category

In [0]:
df = (
    df
    .withColumn(
        "temperature_category",
        when(
            col("temperature") < 15,
            "Cold"
        )
        .when(
            col("temperature") < 30,
            "Moderate"
        )
        .otherwise(
            "Hot"
        )
    )
)

## create humidity category

In [0]:
df = (
    df
    .withColumn(
        "humidity_category",
        when(
            col("humidity") < 40,
            "Dry"
        )
        .when(
            col("humidity") < 70,
            "Normal"
        )
        .otherwise(
            "Humid"
        )
    )
)


##create wind category

In [0]:
df = (
    df
    .withColumn(
        "wind_category",
        when(
            col("wind_speed") < 3,
            "Low"
        )
        .when(
            col("wind_speed") < 8,
            "Medium"
        )
        .otherwise(
            "High"
        )
    )
)


##data checks

In [0]:
df.filter(
    col("city").isNull()
).show()


In [0]:
df.filter(
    col("temperature").isNull()
).show()

In [0]:
df.printSchema()

##silver Path

In [0]:
silver_path = (
    "s3://weather-data-pipeline-235/silver/weather/"
)

In [0]:
(
    df.write
    .mode("overwrite")
    .partitionBy(
        "year",
        "month",
        "day"
    )
    .parquet(
        silver_path
    )
)

Why partition?
Future queries:
SELECT *
FROM silver_weather
WHERE year=2026
AND month=06
AND day=05
will scan only required partitions.

In [0]:
silver_df = (
    spark.read.parquet(
        silver_path
    )
)

In [0]:
display(silver_df.limit(5))

In [0]:
silver_df.printSchema()

## to check changes